# Diabetes Health Indicators: EDA and Bias Audit

This notebook explores the 2015 BRFSS diabetes health-indicators data and audits representation across demographic groups. The binary target uses `0` for no diabetes and `1` for prediabetes or diabetes.

## 1. Load the data

The binary dataset is used for this EDA and bias audit. The three-class dataset is also loaded for future analysis that separates prediabetes from diabetes.

In [8]:
import pandas as pd

df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
df_data = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')

print('Binary dataset shape:', df.shape)
display(df.head())

Binary dataset shape: (253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## 2. Define variables and labels

Columns are grouped by data type to make it easier to choose suitable summaries and visualizations. Demographic columns are used later in the bias audit.

In [9]:
target_col = 'Diabetes_binary'
binary_cols = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
               'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
               'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex']
ordinal_cols = ['GenHlth', 'Age', 'Education', 'Income']
numeric_cols = ['BMI', 'MentHlth', 'PhysHlth']
demographic_cols = ['Sex', 'Age', 'Education', 'Income']
predictor_cols = binary_cols + ordinal_cols + numeric_cols

column_labels = {
    'Diabetes_binary': 'Diabetes Status', 'HighBP': 'High Blood Pressure', 'HighChol': 'High Cholesterol',
    'CholCheck': 'Cholesterol Check', 'BMI': 'Body Mass Index', 'Smoker': 'Smoker', 'Stroke': 'Stroke',
    'HeartDiseaseorAttack': 'Heart Disease or Heart Attack', 'PhysActivity': 'Physical Activity',
    'Fruits': 'Consumes Fruit', 'Veggies': 'Consumes Vegetables', 'HvyAlcoholConsump': 'Heavy Alcohol Consumption',
    'AnyHealthcare': 'Has Healthcare', 'NoDocbcCost': 'Skipped Doctor Due to Cost', 'GenHlth': 'General Health',
    'MentHlth': 'Days of Poor Mental Health', 'PhysHlth': 'Days of Poor Physical Health',
    'DiffWalk': 'Difficulty Walking', 'Sex': 'Sex', 'Age': 'Age Category',
    'Education': 'Education Level', 'Income': 'Income Level'
}
target_labels = {0.0: 'No diabetes', 1.0: 'Prediabetes or diabetes'}

print(f'Number of potential predictors: {len(predictor_cols)}')
print('Demographic columns for the bias audit:', demographic_cols)

Number of potential predictors: 21
Demographic columns for the bias audit: ['Sex', 'Age', 'Education', 'Income']


## 3. Data quality and descriptive statistics

Before interpreting the data, check summary statistics, missing values, and whether the expected columns are present. The dataset is intended to be pre-cleaned, so the missing-value total should be zero.

In [10]:
def get_summary_stats(data):
    return data.describe()

def check_missing_values(data):
    return data.isnull().sum()

def check_total_missing_values(data):
    return data.isnull().sum().sum()

def check_column_groups(data):
    expected_cols = [target_col] + predictor_cols
    return {
        'missing_cols': [col for col in expected_cols if col not in data.columns],
        'extra_cols': [col for col in data.columns if col not in expected_cols]
    }

print('Summary statistics:')
display(get_summary_stats(df))
print('Missing values per column:')
display(check_missing_values(df).to_frame('missing_values'))
print('Total missing values overall:', check_total_missing_values(df))
print('Column group check:', check_column_groups(df))

Summary statistics:


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,...,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000,253680.000000
mean,0.139333,0.429001,0.424121,0.962670,28.382364,0.443169,0.040571,0.094186,0.756544,0.634256,...,0.951053,0.084177,2.511392,3.184772,4.242081,0.168224,0.440342,8.032119,5.050434,6.053875
std,0.346294,0.494934,0.494210,0.189571,6.608694,0.496761,0.197294,0.292087,0.429169,0.481639,...,0.215759,0.277654,1.068477,7.412847,8.717951,0.374066,0.496429,3.054220,0.985774,2.071148
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,5.000000
50%,0.000000,0.000000,0.000000,1.000000,27.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,7.000000
75%,0.000000,1.000000,1.000000,1.000000,31.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,2.000000,3.000000,0.000000,1.000000,10.000000,6.000000,8.000000
max,1.000000,1.000000,1.000000,1.000000,98.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000


Missing values per column:


,missing_values
Diabetes_binary,0
HighBP,0
HighChol,0
CholCheck,0
BMI,0
Smoker,0
Stroke,0
HeartDiseaseorAttack,0
PhysActivity,0
Fruits,0


Total missing values overall: 0
Column group check: {'missing_cols': [], 'extra_cols': []}


## 4. Potential outliers

This check uses the 1.5 × IQR rule to flag unusually distant numeric values. A flagged value is not necessarily an error; it should be reviewed in context before it is removed or changed.

In [11]:
def find_extreme_outliers_iqr(data, col):
    q1, q3 = data[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return data[(data[col] < lower_bound) | (data[col] > upper_bound)]

def get_outlier_counts(data, target):
    columns = data.drop(columns=[target]).select_dtypes(include='number').columns
    counts = {col: len(find_extreme_outliers_iqr(data, col)) for col in columns}
    return pd.Series(counts).sort_values(ascending=False)

print('Potential outlier counts by predictor column:')
display(get_outlier_counts(df, target_col).to_frame('outlier_count'))

Potential outlier counts by predictor column:


,outlier_count
PhysActivity,61760
Veggies,47839
DiffWalk,42675
PhysHlth,40949
MentHlth,36208
HeartDiseaseorAttack,23893
NoDocbcCost,21354
HvyAlcoholConsump,14256
AnyHealthcare,12417
GenHlth,12081


## 5. Target distribution and predictor summaries

The target distribution reveals whether diabetes status is imbalanced. The summary tables show the observed diabetes rate at each value of a predictor, which can later be used for bar or line charts.

In [12]:
def summarize_target_by_column(data, col, target):
    return (data.groupby(col)[target].agg(['count', 'mean']).reset_index()
            .rename(columns={col: 'value', 'count': 'response_count', 'mean': 'diabetes_rate'}))

def get_all_target_summaries(data, columns, target):
    return {col: summarize_target_by_column(data, col, target) for col in columns}

print('Target class counts:')
display(df[target_col].value_counts().rename(index=target_labels).to_frame('count'))
print('Target class proportions:')
display(df[target_col].value_counts(normalize=True).rename(index=target_labels).to_frame('proportion'))

target_summaries = get_all_target_summaries(df, predictor_cols, target_col)
print('Example visualization-ready summary: diabetes rate by HighBP')
display(target_summaries['HighBP'])

Target class counts:


,count
Diabetes_binary,
No diabetes,218334
Prediabetes or diabetes,35346


Target class proportions:


,proportion
Diabetes_binary,
No diabetes,0.860667
Prediabetes or diabetes,0.139333


Example visualization-ready summary: diabetes rate by HighBP


,value,response_count,diabetes_rate
0,0.0,144851,0.060352
1,1.0,108829,0.244457


## 6. Correlations with diabetes status

Correlation measures the strength and direction of linear relationships. It helps prioritize variables for further exploration, but it does not prove that one variable causes diabetes.

In [13]:
def get_correlation_matrix(data, columns):
    return data[columns].corr()

correlation_matrix = get_correlation_matrix(df, predictor_cols + [target_col])
print('Correlation with target variable:')
display(correlation_matrix[target_col].sort_values(ascending=False).to_frame('correlation'))

Correlation with target variable:


,correlation
Diabetes_binary,1.000000
GenHlth,0.293569
HighBP,0.263129
DiffWalk,0.218344
BMI,0.216843
HighChol,0.200276
Age,0.177442
HeartDiseaseorAttack,0.177282
PhysHlth,0.171337
Stroke,0.105816


## 7. Demographic bias audit

Each audit table shows a demographic group's sample count, share of the dataset, and observed diabetes rate. Groups with small counts or substantially different representation should receive extra care in later modeling and evaluation.

In [14]:
def check_group_representation(data, group_col):
    return data[group_col].value_counts(normalize=True).sort_index()

def compare_diabetes_rate_by_group(data, group_col, target):
    return data.groupby(group_col)[target].mean().sort_index()

def bias_audit_by_group(data, group_col, target):
    return pd.DataFrame({
        'response_count': data[group_col].value_counts().sort_index(),
        'group_representation': check_group_representation(data, group_col),
        'diabetes_rate': compare_diabetes_rate_by_group(data, group_col, target)
    })

def get_all_bias_audits(data, group_cols, target):
    return {col: bias_audit_by_group(data, col, target) for col in group_cols}

bias_audits = get_all_bias_audits(df, demographic_cols, target_col)
for col in demographic_cols:
    print(f'Bias audit by {column_labels[col]}:')
    display(bias_audits[col])

Bias audit by Sex:


,response_count,group_representation,diabetes_rate
Sex,,,
0.0,141974,0.559658,0.129679
1.0,111706,0.440342,0.151603


Bias audit by Age Category:


,response_count,group_representation,diabetes_rate
Age,,,
1.0,5700,0.022469,0.013684
2.0,7598,0.029951,0.018426
3.0,11123,0.043847,0.028230
4.0,13823,0.054490,0.045287
5.0,16157,0.063690,0.065049
6.0,19819,0.078126,0.087895
7.0,26314,0.103729,0.117352
8.0,30832,0.121539,0.138265
9.0,33244,0.131047,0.172452


Bias audit by Education Level:


,response_count,group_representation,diabetes_rate
Education,,,
1.0,174,0.000686,0.270115
2.0,4043,0.015937,0.292605
3.0,9478,0.037362,0.242245
4.0,62750,0.247359,0.176351
5.0,69910,0.275583,0.148105
6.0,107325,0.423072,0.096902


Bias audit by Income Level:


,response_count,group_representation,diabetes_rate
Income,,,
1.0,9811,0.038675,0.242891
2.0,11783,0.046448,0.261903
3.0,15994,0.063048,0.223084
4.0,20135,0.079372,0.201341
5.0,25883,0.102030,0.174014
6.0,36470,0.143764,0.145078
7.0,43219,0.170368,0.121821
8.0,90385,0.356295,0.079604


## 8. Visualization-ready results

These outputs are collected in one dictionary for use in future charts or models.

In [15]:
visualization_data = {
    'target_col': target_col, 'predictor_cols': predictor_cols,
    'binary_cols': binary_cols, 'ordinal_cols': ordinal_cols, 'numeric_cols': numeric_cols,
    'demographic_cols': demographic_cols, 'column_labels': column_labels,
    'target_labels': target_labels, 'target_summaries': target_summaries,
    'correlation_matrix': correlation_matrix, 'bias_audits': bias_audits
}
print('Visualization data is ready.')

Visualization data is ready.
